# Step 02 — Tile Downloading & Previewing

This notebook downloads map tiles from Earth Engine using the standard Web Mercator tile scheme (also called XYZ tiles). In this scheme, the world is divided into a pyramid of square tiles for each zoom level. A tile is uniquely identified by its zoom (`z`), column (`x`), and row (`y`). Because many providers adopt this same addressing system, you can fetch imagery for the same location across different services by requesting the same `z/x/y` tile coordinates. This uniformity is useful for:

- **Web mapping**: Browsers and libraries like Leaflet or MapLibre request tiles by `z/x/y`, so maps from different sources align perfectly without extra transformations.
- **Analytics and ML**: Each tile's filename encodes its geographic footprint. You can convert a tile path (e.g., `z/x/y`) back to a bounding box or center latitude/longitude with utilities like `mercantile`, making it easy to pair pixels with real-world locations for training data.

In this notebook, we:
- Compute the set of `z/x/y` tiles that cover an Area of Interest (AOI) from GeoJSON.
- Request those tiles from Earth Engine, which serves JPEG imagery (even though the URLs omit file extensions).
- Save each tile directly as `.jpg` using the same `z/x/y` folder structure so it stays compatible with web maps and downstream ML workflows.

## 1. Install Necessary Packages

These packages provide the core tools we use in this notebook:
- **requests** for fetching tiles over HTTP.
- **mercantile** to translate between geographic bounds and XYZ tile coordinates.
- **pandas/geopandas/fiona** to read CSV and GeoJSON inputs.
- **Pillow (PIL)** to inspect image responses when needed.
- **matplotlib/folium** to preview tiles and the AOI on a map.

In [ ]:
!pip install -q --upgrade mercantile fiona geopandas pandas pillow requests folium matplotlib

## 2. Imports

This code block loads the libraries used throughout the notebook:
- Data handling (`pandas`, `geopandas`)
- Paths and JSON utilities
- Mapping helpers (`folium`, `mercantile`)
- Plotting (`matplotlib`)
- HTTP requests (`requests`) and image handling (`PIL`)
- Concurrency (`ThreadPoolExecutor`) for parallel tile downloads

In [ ]:
import pandas as pd              # Tabular data (CSV of tile services)
import geopandas as gpd           # Geospatial data (GeoJSON AOI)
from pathlib import Path          # Filesystem paths (cross-platform)
import folium                     # Lightweight web map previews
import mercantile                 # XYZ tile math (bbox → tiles, tiles → bbox)
import json                       # Reading raw GeoJSON
import matplotlib.pyplot as plt   # Plotting previews of tiles
from PIL import Image             # Image inspection when previewing tiles
import requests                   # HTTP requests to fetch tiles
from io import BytesIO            # Wrap raw bytes so PIL can open them
from concurrent.futures import ThreadPoolExecutor, as_completed  # Parallel downloads
from shapely.geometry import box  # Build tile polygons for intersection test
from shapely.ops import unary_union  # Dissolve AOI features into one geometry

## 3. Define Download Helper Function

The `download_earthengine_tiles()` function handles parallel tile downloads with browser User-Agent mimicking and retry logic. Earth Engine serves JPEG imagery even though the URLs omit file extensions, so tiles are saved directly as `.jpg`.

In [ ]:
# Earth Engine tile downloader - mimics browser, includes retry logic
import time

def download_earthengine_tiles(url_template: str, save_dir: Path, tiles_list, num_workers: int = 10) -> dict:
    """
    Download Earth Engine tiles and save directly as JPG.
    Earth Engine serves JPEG even without an extension in the URL.
    We mimic a browser User-Agent and retry a few times to reduce blank tiles.
    """
    headers = {
        # Pretend to be a modern browser to avoid simple blocks
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    downloaded = 0  # Counter for successful tiles
    failed = 0      # Counter for tiles that still fail after retries
    max_retries = 3

    def download_single_tile(tile):
        """Fetch one tile with retries and save as JPG."""
        nonlocal downloaded, failed
        url = url_template.replace('{z}', str(tile.z)).replace('{x}', str(tile.x)).replace('{y}', str(tile.y))

        for attempt in range(max_retries):
            try:
                time.sleep(0.1)  # Gentle pacing to avoid rate limits

                response = requests.get(url, headers=headers, timeout=30)
                response.raise_for_status()

                # Skip obviously bad/blank responses
                if len(response.content) < 100:
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
                    else:
                        raise Exception("Blank tile response")

                # Create z/x directory and save tile as .jpg directly (no conversion)
                tile_dir = save_dir / str(tile.z) / str(tile.x)
                tile_dir.mkdir(parents=True, exist_ok=True)
                tile_path = tile_dir / f"{tile.y}.jpg"
                tile_path.write_bytes(response.content)

                downloaded += 1
                return

            except Exception:
                if attempt == max_retries - 1:
                    failed += 1
                    print(f"    Failed tile {tile.z}/{tile.x}/{tile.y} after {max_retries} attempts")
                else:
                    time.sleep(1)  # Back off briefly before retrying

    print(f"  Downloading {len(tiles_list)} tiles with browser mimicking...")
    # Launch parallel downloads
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(download_single_tile, tile) for tile in tiles_list]
        for future in as_completed(futures):
            future.result()  # Propagate any exceptions

    return {"downloaded": downloaded, "failed": failed}

## 4. Parameters

This block collects all user-editable settings in one place so you only change values here:
- `project`: folder name under `../tilesets/` where downloads are stored.
- `tile_csv_path`: CSV listing Earth Engine tile URL templates (with `{z}`, `{x}`, `{y}` placeholders).
- `geojson_path`: GeoJSON file that defines the Area of Interest (AOI).
- `zoom_level`: tile zoom to request (higher zoom = more tiles, smaller ground footprint).
- `num_workers`: how many concurrent downloads to run.

In [ ]:
# ---- User parameters (edit these) ----
project = "mortalitree_czu_download_Z18"   # Output folder name under ../tilesets/

tile_csv_path = "../output/tile_urls.csv"  # CSV generated by Step 01 (name, tile_url columns)
geojson_path = "../data/CZU_Fire_Perimeter_-2212360052269988636.geojson"  # AOI GeoJSON file

zoom_level = 18   # Tile zoom level (higher = more detail, more tiles)
num_workers = 40  # Parallel download threads
# --------------------------------------

## 5. Build Tile Service List

We read the CSV of tile services and build a list of dictionaries, each holding a service name and its URL template. This keeps service info organized and easy to filter later.

In [ ]:
# Read the services CSV
df = pd.read_csv(tile_csv_path)  # Load service names and URL templates

# Build a list of dicts: [{name, url}, ...]
tile_service_urls = []
for _, row in df.iterrows():
    tile_service_urls.append({
        'name': row['name'],    # Friendly service name
        'url': row['tile_url']  # Earth Engine URL template with {z}/{x}/{y}
    })

# Quick preview to confirm we parsed correctly
print(f"Found {len(tile_service_urls)} tile services:\n")
for i, service in enumerate(tile_service_urls):  # Show services
    print(f"{i}. {service['name']}")
    print(f"   URL: {service['url']}\n")

## 6. Create the Area of Interest (AOI) and Compute Tile List

We load the GeoJSON AOI, dissolve all features into a single geometry, compute the bounding-box candidate tiles, then **filter to only the tiles that actually intersect the AOI** (not just the bounding box). This avoids downloading tiles that fall in the corners of the bbox but outside the real features.

In [ ]:
# Load AOI GeoJSON from disk
with open(geojson_path, 'r') as f:
    geojson_data = json.load(f)

# Read into GeoDataFrame for convenience (enables bounds and CRS handling)
gdf = gpd.read_file(str(geojson_path))

# Bounding box: [minx, miny, maxx, maxy]
AOI = gdf.total_bounds
print(f"AOI Bounding Box: {AOI}")
print(f"West: {AOI[0]}, South: {AOI[1]}, East: {AOI[2]}, North: {AOI[3]}")

# Dissolve all AOI features into a single geometry for intersection testing
aoi_union = unary_union(gdf.geometry)

# Generate candidate tiles from the bounding box, then filter by actual AOI intersection
west, south, east, north = AOI
bbox_tiles = list(mercantile.tiles(west, south, east, north, zooms=zoom_level))

aoi_tiles = []
for tile in bbox_tiles:
    t_bounds = mercantile.bounds(tile)
    t_geom = box(t_bounds.west, t_bounds.south, t_bounds.east, t_bounds.north)
    if aoi_union.intersects(t_geom):
        aoi_tiles.append(tile)

print(f"\nBounding-box candidates: {len(bbox_tiles)} tiles")
print(f"After AOI intersection:  {len(aoi_tiles)} tiles  ({len(bbox_tiles) - len(aoi_tiles)} removed)")

# Center point for map preview
center_lat = (AOI[1] + AOI[3]) / 2
center_lon = (AOI[0] + AOI[2]) / 2

# Create a simple Folium map to confirm the AOI
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
folium.GeoJson(geojson_data, name="AOI Features").add_to(m)

# Draw the bounding box as a rectangle
bbox_rectangle = folium.Rectangle(
    bounds=[[AOI[1], AOI[0]], [AOI[3], AOI[2]]],
    color="red",
    fill=True,
    fill_opacity=0.1,
    weight=2,
)
bbox_rectangle.add_to(m)

# Display interactive map in the notebook
m

## 7. Create a Download List

Pick which services to download. We keep it as a Python list so you can select all services or a subset without changing the rest of the notebook.

In [ ]:
# Choose which services to download
# Examples are shown below—only one option should be active at a time.

# Option 1: Download ALL services from the CSV
download_list = tile_service_urls

# Option 2: Download specific services by index (example uses first, fifth, sixth, seventh)
# download_list = [tile_service_urls[0], tile_service_urls[4], tile_service_urls[5], tile_service_urls[6]]

# Option 2 alternative: single service by index (wrap in list to keep downstream loops happy)
#download_list = [tile_service_urls[15]]

# Option 3: Download services by name
# download_list = [s for s in tile_service_urls if s['name'] in ['service_a', 'service_b']]

print(f"Services selected for download: {len(download_list)}")
for service in download_list:
    print(f"  - {service['name']}")


## 8. Prepare Output Folders

Create the output directory under `../tilesets/<project>/` and set guard flags. If no services are selected the remaining cells skip gracefully.

In [ ]:
# Prepare output folders and guardrails
# This creates ../tilesets/<project>/ if it does not already exist and sets flags for later steps.
base_save_dir = Path("../tilesets") / project
base_save_dir.mkdir(parents=True, exist_ok=True)

# These keep state for later cells
service_status = []  # Will hold per-service results for summary and preview
skip_download = False  # When True, later cells skip work instead of erroring

# If nothing is selected, stop early so we do not run empty work in later steps
if not download_list:
    print("No services selected for download. Skipping download, summary, and preview.")
    skip_download = True
else:
    print(f"Services selected for download: {len(download_list)}")
    print(f"Tiles will be saved under: {base_save_dir.resolve()}")

## 9. Download Tiles for Each Service

For every chosen service we:
- Build the list of `z/x/y` tiles covering the AOI at the chosen zoom.
- Download tiles in parallel (directly as `.jpg`) using the browser-mimicking downloader.
- Record successes/failures to use in the summary and preview steps.

In [ ]:
import sys

# Download tiles in parallel for each service
# This loop downloads tiles service by service, but each service's tiles are fetched in parallel threads.
if skip_download:
    print("Skipping downloads because no services were selected.")
else:
    for service in download_list:
        print(f"\n{'='*60}")
        print(f"Downloading tiles for: {service['name']}")
        print(f"{'='*60}")

        # Prepare per-service save folder, e.g., ../tilesets/project/ServiceName
        url_template = service['url']  # Earth Engine URL template with {z}/{x}/{y}
        save_dir = base_save_dir / service['name']
        save_dir.mkdir(parents=True, exist_ok=True)

        # Use the pre-filtered tile list (only tiles that intersect the actual AOI geometry)
        print(f"Downloading {len(aoi_tiles)} tiles at zoom level {zoom_level} (AOI-filtered)...")

        try:
            # Track progress live as tiles complete
            
            downloaded_count = 0
            failed_count = 0
            total_tiles = len(aoi_tiles)
            
            def download_single_tile_with_progress(tile):
                nonlocal downloaded_count, failed_count
                url = url_template.replace('{z}', str(tile.z)).replace('{x}', str(tile.x)).replace('{y}', str(tile.y))
                
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
                }
                
                max_retries = 3
                for attempt in range(max_retries):
                    try:
                        time.sleep(0.1)
                        response = requests.get(url, headers=headers, timeout=30)
                        response.raise_for_status()
                        
                        if len(response.content) < 100:
                            if attempt < max_retries - 1:
                                time.sleep(1)
                                continue
                            else:
                                failed_count += 1
                                return
                        
                        tile_dir = save_dir / str(tile.z) / str(tile.x)
                        tile_dir.mkdir(parents=True, exist_ok=True)
                        tile_path = tile_dir / f"{tile.y}.jpg"
                        tile_path.write_bytes(response.content)
                        
                        downloaded_count += 1
                        
                        # Print progress bar
                        completed = downloaded_count + failed_count
                        percent = int((completed / total_tiles) * 100)
                        bar_length = 50
                        filled = int((completed / total_tiles) * bar_length)
                        bar = '=' * filled + '>' + '.' * (bar_length - filled - 1)
                        sys.stdout.write(f'\r  [{bar}] {percent}% ({completed}/{total_tiles}) OK: {downloaded_count}, Failed: {failed_count}')
                        sys.stdout.flush()
                        return
                        
                    except Exception:
                        if attempt == max_retries - 1:
                            failed_count += 1
                        else:
                            time.sleep(1)
            
            # Launch parallel downloads
            with ThreadPoolExecutor(max_workers=num_workers) as executor:
                futures = [executor.submit(download_single_tile_with_progress, tile) for tile in aoi_tiles]
                for future in as_completed(futures):
                    future.result()
            
            print()  # New line after progress bar
            print(f"✓ Downloaded {downloaded_count} tiles as JPG", end="")
            if failed_count > 0:
                print(f" ({failed_count} failed)")
            else:
                print()

            # Record success for later summary and preview
            service_status.append({"name": service['name'], "status": "success", "tiles": downloaded_count})
        except Exception as exc:
            # Record failure so later steps can report it
            print(f"✗ Failed download for {service['name']}: {exc}")
            service_status.append({"name": service['name'], "status": "failed", "error": str(exc)})
            continue

    print("\nFinished running downloads for all selected services.")

## 10. Summarize Download Results

After the downloads finish, print a quick report so you can see which services succeeded or failed before previewing tiles.

In [ ]:
# Print a summary of successes and failures
if skip_download:
    print("No services were selected, so there is nothing to summarize.")
elif not service_status:
    print("Downloads did not run or recorded no results.")
else:
    print(f"\n{'='*60}")
    print("Download summary:")
    for entry in service_status:
        if entry["status"] == "success":
            print(f"  ✓ {entry['name']} ({entry['tiles']} tiles)")
        else:
            print(f"  ✗ {entry['name']} -- {entry['status']} ({entry.get('error', 'no error reported')})")
    print(f"{'='*60}")

## 11. Preview Center Tiles

Plot one tile from the center of the AOI for each service. This is a quick visual check that the downloads look right and that files were saved where expected.

In [ ]:
# Visual preview of one tile per service (center of AOI)
if skip_download:
    print("Skipping preview because no services were selected.")
else:
    # Compute the center tile once; it is reused for all services
    center_lat = (AOI[1] + AOI[3]) / 2
    center_lon = (AOI[0] + AOI[2]) / 2
    center_tile = mercantile.tile(center_lon, center_lat, zoom_level)

    # Set up a plot column that matches the number of selected services
    num_services = len(download_list)
    if num_services == 0:
        print("No services available for preview.")
    else:
        fig, axes = plt.subplots(1, num_services, figsize=(5*num_services, 5))
        if num_services == 1:
            axes = [axes]  # Normalize to list for consistent indexing

        status_map = {entry['name']: entry for entry in service_status}

        for idx, service in enumerate(download_list):
            status = status_map.get(service['name'], {"status": "unknown"})

            # If this service did not download successfully, show a text placeholder
            if status.get("status") != "success":
                axes[idx].text(0.5, 0.5, f"No preview\n{service['name']}\n{status.get('status')}",
                               ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')
                continue

            # Construct expected tile path: ../tilesets/project/Service/z/x/y.jpg
            tile_path = base_save_dir / service['name'] / str(center_tile.z) / str(center_tile.x) / f"{center_tile.y}.jpg"

            if tile_path.exists():
                img = Image.open(tile_path)
                axes[idx].imshow(img)
                axes[idx].set_title(service['name'], fontsize=10)
                axes[idx].axis('off')
            else:
                axes[idx].text(0.5, 0.5, f"Tile not found\n{service['name']}",
                               ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')

        plt.tight_layout()
        plt.show()

## 12. Print Downloaded Services

Print the list of successfully downloaded `name` / `tile_url` pairs for reference and easy copy-paste.

In [ ]:
# Print list of downloaded services for reference
print("Downloaded services:\n")

# Check if download_list exists in the global scope (i.e., has been defined in a previous cell)
if "download_list" not in globals():
    print("download_list is not defined. Please run the cell that creates it (\"Create a download list from the dictionary\").")
# Check if service_status exists (this variable is created during the download process)
elif "service_status" not in globals():
    print("service_status is not defined. Please run the download cell before listing services.")
else:
    # Flag to track if we found any successfully downloaded services
    any_success = False
    
    # Loop through each service in the download list
    for service in download_list:
        # Look for this service's status in the service_status list
        # next() finds the first matching dictionary where the name matches
        status = next((s for s in service_status if s.get("name") == service["name"]), None)
        
        # Only print services that downloaded successfully
        if status and status.get("status") == "success":
            # Print the service name (formatted for easy copy-paste)
            print(f"  name: '{service['name']}'")
            # Print the URL template for this service
            print(f"  url: {service['url']}")
            # Add blank line for readability between services
            print()
            # Mark that we found at least one successful service
            any_success = True
    
    # If no services succeeded, let the user know
    if not any_success:
        print("No successfully downloaded services to list.")

## 13. Visual Coverage Check — Downsampled Tile Mosaic

Build a single downsampled mosaic image from the downloaded tiles to quickly verify coverage. Each 256×256 tile is shrunk to a small thumbnail (default 4×4 px) and placed into one composite image. **Missing tiles are filled in red** so gaps are immediately visible.

The mosaic is displayed as a single `ImageOverlay` on leafmap — no tile server needed, loads instantly regardless of tile count.

In [5]:
import leafmap
import numpy as np
import re
from ipyleaflet import ImageOverlay
from pathlib import Path
from PIL import Image
import mercantile

# ---- Settings -------------------------------------------------------------
view_index = 0          # <-- change this to view a different service
thumb_px   = 4          # pixels per tile in the mosaic (4 = fast, 8 = sharper)

# Define base_save_dir - update this path to match your download location
base_save_dir = Path("../tilesets/mortalitree_czu_download_Z18")  # <-- adjust this path as needed

# ---- 1. Discover downloaded services from disk ----------------------------
service_dirs = sorted([
    d.name for d in base_save_dir.iterdir()
    if d.is_dir() and any(d.rglob("*.jpg"))
])
print(f"Found {len(service_dirs)} downloaded services under {base_save_dir}:")
for i, name in enumerate(service_dirs):
    jpg_count = len(list((base_save_dir / name).rglob("*.jpg")))
    print(f"  {i:2d}. {name}  ({jpg_count} tiles)")

view_service = service_dirs[view_index]
service_dir  = base_save_dir / view_service

# Parse NAIP year and band type
year_match = re.search(r'(\d{4})', view_service)
naip_year  = year_match.group(1) if year_match else 'Unknown'
band_type  = 'IR (Near-Infrared)' if '_ir_' in view_service else 'RGB (Natural Color)'

# ---- 2. Compute the tile grid extents ------------------------------------
# Ensure required variables exist when this cell is executed standalone.
if 'aoi_tiles' not in globals() or not aoi_tiles:
    # Infer zoom_level from folder structure if needed.
    if 'zoom_level' not in globals():
        zoom_dirs = sorted(
            [p for p in service_dir.iterdir() if p.is_dir() and p.name.isdigit()],
            key=lambda p: int(p.name)
        )
        if not zoom_dirs:
            raise ValueError(f"No zoom directories found in {service_dir}")
        zoom_level = int(zoom_dirs[0].name)
        print(f"Inferred zoom_level={zoom_level} from disk")

    tile_files = list((service_dir / str(zoom_level)).rglob("*.jpg"))
    inferred_tiles = []
    for p in tile_files:
        try:
            x = int(p.parent.name)
            y = int(p.stem)
            inferred_tiles.append(mercantile.Tile(x=x, y=y, z=zoom_level))
        except ValueError:
            continue

    if not inferred_tiles:
        raise ValueError(f"No valid tiles found under {service_dir / str(zoom_level)}")

    aoi_tiles = inferred_tiles
    print(f"Inferred aoi_tiles from disk: {len(aoi_tiles)} tiles")
elif 'zoom_level' not in globals():
    zoom_level = aoi_tiles[0].z

# Use the expected aoi_tiles list so missing files show up as red
xs = [t.x for t in aoi_tiles]
ys = [t.y for t in aoi_tiles]
min_x, max_x = min(xs), max(xs)
min_y, max_y = min(ys), max(ys)
cols = max_x - min_x + 1
rows = max_y - min_y + 1

print(f"\nViewing: {view_service}")
print(f"NAIP Year: {naip_year}  |  Band Type: {band_type}")
print(f"Grid: {cols} cols × {rows} rows = {cols * rows} cells")

# ---- 3. Build the downsampled mosaic -------------------------------------
# Start with a red image so missing tiles are immediately visible
mosaic_w = cols * thumb_px
mosaic_h = rows * thumb_px
mosaic = np.full((mosaic_h, mosaic_w, 4), [255, 0, 0, 180], dtype=np.uint8)  # RGBA red

downloaded_count = 0
missing_count = 0
expected_set = {(t.x, t.y) for t in aoi_tiles}

for tile in aoi_tiles:
    col = tile.x - min_x
    row = tile.y - min_y
    y0 = row * thumb_px
    x0 = col * thumb_px

    tile_path = service_dir / str(zoom_level) / str(tile.x) / f"{tile.y}.jpg"
    if tile_path.exists():
        try:
            img = Image.open(tile_path)
            # Downsample to thumbnail size using fast box filter
            thumb = img.resize((thumb_px, thumb_px), Image.LANCZOS)
            arr = np.array(thumb)
            # Handle grayscale images
            if arr.ndim == 2:
                arr = np.stack([arr, arr, arr], axis=-1)
            # Write RGB + full alpha into the mosaic
            mosaic[y0:y0+thumb_px, x0:x0+thumb_px, :3] = arr[:, :, :3]
            mosaic[y0:y0+thumb_px, x0:x0+thumb_px, 3]  = 255
            downloaded_count += 1
        except Exception:
            missing_count += 1  # corrupt file — stays red
    else:
        missing_count += 1

print(f"Tiles found: {downloaded_count}  |  Missing/corrupt: {missing_count}")

# ---- 4. Save mosaic as a temporary PNG -----------------------------------
mosaic_img = Image.fromarray(mosaic, 'RGBA')
mosaic_path = Path("../output") / f"coverage_{view_service}.png"
mosaic_path.parent.mkdir(parents=True, exist_ok=True)
mosaic_img.save(str(mosaic_path))
print(f"Mosaic saved: {mosaic_path}  ({mosaic_w}×{mosaic_h} px)")

# ---- 5. Compute geographic bounds of the mosaic --------------------------
# Top-left corner of the top-left tile → bottom-right corner of the bottom-right tile
tl_bounds = mercantile.bounds(mercantile.Tile(min_x, min_y, zoom_level))
br_bounds = mercantile.bounds(mercantile.Tile(max_x, max_y, zoom_level))
image_bounds = [
    [br_bounds.south, tl_bounds.west],   # southwest
    [tl_bounds.north, br_bounds.east],   # northeast
]

center_lat = (tl_bounds.north + br_bounds.south) / 2
center_lon = (tl_bounds.west  + br_bounds.east)  / 2

# ---- 6. Display on leafmap -----------------------------------------------
m = leafmap.Map(center=[center_lat, center_lon], zoom=12)

# Encode image as base64 data URI so it loads instantly (no server needed)
import base64
with open(str(mosaic_path), 'rb') as f:
    b64 = base64.b64encode(f.read()).decode('ascii')
data_uri = f"data:image/png;base64,{b64}"

overlay = ImageOverlay(
    url=data_uri,
    bounds=image_bounds,
    name=f"{view_service} coverage",
    opacity=0.9,
)
m.add(overlay)

# Overlay the AOI boundary
if 'gdf' in globals() and gdf is not None:
    m.add_gdf(
        gdf,
        layer_name="AOI Boundary",
        style={"color": "yellow", "weight": 2, "fillOpacity": 0.0}
    )
else:
    # Fallback: draw boundary from computed image bounds so this cell runs standalone
    sw_lat, sw_lon = image_bounds[0]
    ne_lat, ne_lon = image_bounds[1]
    boundary_geojson = {
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "properties": {"name": "AOI Boundary (inferred)"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [sw_lon, sw_lat],
                    [ne_lon, sw_lat],
                    [ne_lon, ne_lat],
                    [sw_lon, ne_lat],
                    [sw_lon, sw_lat]
                ]]
            }
        }]
    }
    m.add_geojson(
        boundary_geojson,
        layer_name="AOI Boundary (inferred)",
        style={"color": "yellow", "weight": 2, "fillOpacity": 0.0}
    )

print(f"\nRed areas = missing tiles. Yellow outline = AOI boundary.")
m

Found 4 downloaded services under ../tilesets/mortalitree_czu_download_Z18:
   0. gee_naip_2020  (27904 tiles)
   1. gee_naip_2022  (22897 tiles)
   2. gee_naip_ir_2020  (22596 tiles)
   3. gee_naip_ir_2022  (23143 tiles)

Viewing: gee_naip_2020
NAIP Year: 2020  |  Band Type: RGB (Natural Color)
Grid: 189 cols × 233 rows = 44037 cells
Tiles found: 27904  |  Missing/corrupt: 0
Mosaic saved: ../output/coverage_gee_naip_2020.png  (756×932 px)

Red areas = missing tiles. Yellow outline = AOI boundary.


Map(center=[37.145432620216745, -122.21672058105469], controls=(ZoomControl(options=['position', 'zoom_in_text…